# DANFront - Banat of Temesvár (First Military Survey): control-point crops

Like the Transylvania DVD, the Temesvár DVD has no georeferencing. This notebook downloads the ~39 sheets that contain a
well-known town, cuts a picture around the place where the town should be (from the overview sheet), and packs the pictures
into one zip. Download the zip and give it to Claude. A normal CPU runtime is enough, no R2 keys and no Drive. ~15 minutes.

In [ ]:
!pip -q install rasterio pyproj
import os, sys
os.makedirs('/content/work', exist_ok=True); os.chdir('/content/work'); sys.path.insert(0, '/content/work')

In [ ]:
%%writefile /content/work/danfront_funcs.py
# ---- helper functions (pure Python; no Colab-specific calls) ----
import csv, json, math, os, shutil, subprocess, sys, time
import requests

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9

def wait_for_disk(path, min_free=12, resume_at=20):
    """Pause while the local disk is nearly full (Drive needs time to sync its write cache)."""
    if free_gb(path) >= min_free:
        return
    print(f"  local disk low ({free_gb(path):.0f} GB free) - waiting for Drive to catch up ...")
    while free_gb(path) < resume_at:
        time.sleep(30)
    print("  continuing")

def download_parts(url, dest_prefix, part_size=4 * 1024**3, chunk=8 * 1024**2, disk_path="/content"):
    """Download url as raw volumes dest_prefix.001, .002, ... (7-Zip opens these directly).
    Finished parts are skipped, so the cell can be re-run with a fresh link after an interruption."""
    s = requests.Session()
    r = s.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    cr = r.headers.get("Content-Range"); status = r.status_code; ctype = r.headers.get("Content-Type", ""); r.close()
    if status in (401, 403) or "text/html" in ctype:
        raise RuntimeError(f"The link was refused (HTTP {status}, {ctype}). It has probably expired: copy a fresh download link and run this cell again.")
    if status != 206 or not cr:
        raise RuntimeError(f"The server did not accept a ranged request (HTTP {status}). Send this message to Claude.")
    total = int(cr.split("/")[-1])
    size_file = dest_prefix + ".size"
    if os.path.exists(size_file):
        old = int(open(size_file).read().strip())
        if old != total:
            raise RuntimeError(f"This link points to a file of {total} bytes, but the parts already downloaded belong to a file of {old} bytes. Wrong link?")
    else:
        open(size_file, "w").write(str(total))
    nparts = math.ceil(total / part_size)
    print(f"archive size {total/1e9:.1f} GB -> {nparts} parts")
    t0 = time.time(); done_bytes = 0; new_bytes = 0
    for k in range(nparts):
        start = k * part_size; end = min(total, start + part_size) - 1; want = end - start + 1
        part = f"{dest_prefix}.{k+1:03d}"
        if os.path.exists(part) and os.path.getsize(part) == want:
            done_bytes += want; continue
        tmp = part + ".tmp"
        for attempt in range(5):
            try:
                r = s.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=120)
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}")
                got = 0
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk):
                        f.write(c); got += len(c)
                        wait_for_disk(disk_path)
                if got != want:
                    raise IOError(f"got {got} of {want} bytes")
                os.replace(tmp, part); break
            except Exception as e:
                print(f"  part {k+1}: {e} - retrying ({attempt+1}/5)"); time.sleep(20)
        else:
            raise RuntimeError("Download keeps failing. The link has probably expired: copy a fresh download link and run this cell again (finished parts are kept).")
        done_bytes += want; new_bytes += want
        rate = new_bytes / max(1, time.time() - t0) / 1e6
        print(f"part {k+1}/{nparts} done  ({done_bytes/1e9:.1f} of {total/1e9:.1f} GB, {rate:.0f} MB/s)")
    have = sum(os.path.getsize(f"{dest_prefix}.{k+1:03d}") for k in range(nparts))
    assert have == total, f"size mismatch: {have} != {total}"
    print("download complete and verified:", f"{total/1e9:.1f} GB")

def list_archive(first_volume, exe="7z"):
    """Return (properties dict, [(path, size)] of GeoTIFFs with coordinates) from the archive listing."""
    out = subprocess.run([exe, "l", "-slt", first_volume], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError("7z could not read the archive:\n" + out.stdout[-2000:] + out.stderr[-2000:])
    head, _, body = out.stdout.partition("----------")
    props = dict(l.split(" = ", 1) for l in head.splitlines() if " = " in l)
    sheets = []
    for block in body.split("\n\n"):
        e = dict(l.split(" = ", 1) for l in block.splitlines() if " = " in l)
        p = e.get("Path", "")
        if not p.lower().endswith((".tif", ".tiff")) or e.get("Folder") == "+" or e.get("Attributes", "").startswith("D"):
            continue
        if "/raw/" in "/" + p.replace("\\", "/"):
            continue                      # RAW sheets (no coordinates) are not for this notebook
        sheets.append((p, int(e.get("Size", 0))))
    return props, sorted(sheets)

def load_done(done_csv):
    if not os.path.exists(done_csv):
        return {}
    with open(done_csv, newline="", encoding="utf-8") as f:
        return {r["path"]: r for r in csv.DictReader(f)}

DONE_FIELDS = ["path", "sheet", "status", "cell_i", "cell_j", "cell_coverage", "tiles"]

def run_batches(cfg, max_sheets=None, batch_size=12, bbox=None, workers=2):
    """Extract -> tile -> upload -> record, batch by batch. Safe to interrupt and re-run."""
    import importlib
    tb = importlib.import_module(cfg.get("tiler", "tiles_2kf_batch"))   # tiles_1kf_batch for the First Survey
    done = load_done(cfg["done_csv"])
    todo = [p for p, _ in cfg["sheets"] if p not in done]
    if max_sheets:
        todo = todo[:max_sheets]
    print(f"{len(done)} sheets already done, {len(todo)} to do in this run")
    if (done or cfg.get("restore_always")) and not os.path.exists(cfg["restored_marker"]):
        print("new session: fetching the tiles made so far back from R2 (needed to merge sheet edges) ...")
        rc = subprocess.run(["rclone", "copy", cfg["remote"], cfg["tiles"], "--transfers", "64", "--checkers", "64"]).returncode
        if rc != 0:
            raise RuntimeError("could not restore tiles from R2")
    os.makedirs(cfg["tiles"], exist_ok=True); open(cfg["restored_marker"], "w").close()
    bounds = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    bounds = {int(z): b for z, b in bounds.items()}
    t0 = time.time(); n_done = 0
    for b0 in range(0, len(todo), batch_size):
        batch = todo[b0:b0 + batch_size]
        wait_for_disk(cfg["disk_path"])
        shutil.rmtree(cfg["extract"], ignore_errors=True); os.makedirs(cfg["extract"])
        if cfg.get("fetch"):              # sheets come as single files (e.g. from a SharePoint folder)
            cfg["fetch"](batch, cfg["extract"])
        else:                             # sheets come out of the 7z archive
            lst = os.path.join(cfg["local"], "batch.txt")
            open(lst, "w", encoding="utf-8").write("\n".join(batch) + "\n")
            ex = subprocess.run([cfg.get("exe", "7z"), "x", cfg["first_volume"], f"-o{cfg['extract']}", f"-i@{lst}", "-y", "-bso0", "-bsp0"],
                                capture_output=True, text=True)
            if ex.returncode not in (0, 1):
                raise RuntimeError("7z extraction failed:\n" + ex.stdout[-1500:] + ex.stderr[-1500:])
        tifs, missing = [], []
        for p in batch:
            fp = os.path.join(cfg["extract"], p.replace("\\", "/"))
            (tifs if os.path.exists(fp) else missing).append((p, fp))
        results = tb.run([fp for _, fp in tifs], cfg["tiles"], 6, 14, workers=workers, bbox=bbox)
        changed = os.path.join(os.path.dirname(os.path.abspath(cfg["tiles"])), "changed_tiles.txt")
        if os.path.getsize(changed) > 0:
            for attempt in range(3):
                rc = subprocess.run(["rclone", "copy", cfg["tiles"], cfg["remote"], "--files-from", changed, "--no-traverse",
                                     "--transfers", "32"]).returncode
                if rc == 0:
                    break
                print("  upload failed, retrying ..."); time.sleep(15)
            else:
                raise RuntimeError("Upload to R2 failed 3 times. Check the R2 keys, then run this cell again.")
        rows = []
        for (p, _), r in zip(tifs, results):
            tb.merge_bounds(bounds, r["bounds"])
            c = r.get("cell", ["", ""])
            rows.append(dict(path=p, sheet=r["sheet"], status=r["status"], cell_i=c[0], cell_j=c[1],
                             cell_coverage=r.get("cell_coverage", ""), tiles=len(r["tiles"])))
        for p, _ in missing:
            rows.append(dict(path=p, sheet=os.path.basename(p), status="ERROR: not extracted", cell_i="", cell_j="", cell_coverage="", tiles=0))
        op = cfg.get("outlines_json") or os.path.join(os.path.dirname(cfg["done_csv"]), "sheet_outlines.json")
        outlines = json.load(open(op)) if os.path.exists(op) else {}
        outlines.update({r["sheet"]: r["outline"] for r in results if r.get("outline") and r["status"].startswith("ok")})
        json.dump(outlines, open(op, "w"))          # sheet name -> corners (lon, lat), for the map's sheet lookup
        new_file = not os.path.exists(cfg["done_csv"])
        with open(cfg["done_csv"], "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS)
            if new_file:
                w.writeheader()
            w.writerows(rows)
        json.dump(bounds, open(cfg["bounds_json"], "w"))
        shutil.rmtree(cfg["extract"], ignore_errors=True)
        n_done += len(batch)
        el = time.time() - t0; left = (len(todo) - n_done) * el / n_done
        bad = [r["sheet"] + ": " + r["status"] for r in rows if not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))]
        print(f"[{n_done}/{len(todo)}] batch done in total {el/60:.0f} min, about {left/60:.0f} min left" + (f"   notes: {bad}" if bad else ""))
    print("run finished")

def stitch_preview(tiles_dir, z, max_px=1600):
    """Stitch all local tiles of zoom z into one image (for a quick look)."""
    import glob
    from PIL import Image
    tiles = glob.glob(os.path.join(tiles_dir, str(z), "*", "*.png"))
    if not tiles:
        return None
    key = lambda p: (int(p.split(os.sep)[-2]), int(os.path.basename(p)[:-4]))
    xs = sorted({key(p)[0] for p in tiles}); ys = sorted({key(p)[1] for p in tiles})
    W = Image.new("RGBA", (256 * (xs[-1] - xs[0] + 1), 256 * (ys[-1] - ys[0] + 1)), (225, 225, 225, 255))
    for p in tiles:
        x, y = key(p); W.alpha_composite(Image.open(p).convert("RGBA"), ((x - xs[0]) * 256, (y - ys[0]) * 256))
    W = W.convert("RGB")
    if max(W.size) > max_px:
        W.thumbnail((max_px, max_px))
    return W

def summary(cfg):
    done = load_done(cfg["done_csv"])
    from collections import Counter
    c = Counter(r["status"].split(":")[0] for r in done.values())
    print("sheets:", len(done), dict(c))
    for r in done.values():
        if (not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))) or "irregular" in r["status"] or "not clipped" in r["status"]:
            print("  ", r["sheet"], "-", r["status"], "coverage", r["cell_coverage"])
    b = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    print("\n--- copy everything below this line to Claude ---")
    print("sheets_ok:", sum(1 for r in done.values() if r["status"].startswith("ok")), "of", len(done))
    print(f"var {cfg.get('bounds_var', 'kf2TileBounds')} = {{")
    for z in sorted(b, key=int):
        v = b[z]; print(f"    {int(z)}: {{ xMin: {v['xMin']}, xMax: {v['xMax']}, yMin: {v['yMin']}, yMax: {v['yMax']} }},")
    print("};")

def repair(cfg):
    """After an interrupted run: fix half-written tiles and put failed sheets back on the to-do list."""
    import glob
    from PIL import Image
    bad = []
    for p in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.png")):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception:
            bad.append(p)
    print(len(bad), "damaged tiles found")
    for p in bad:
        rel = os.path.relpath(p, cfg["tiles"]).replace(os.sep, "/")
        os.remove(p)
        subprocess.run(["rclone", "copyto", f"{cfg['remote']}/{rel}", p], capture_output=True)   # last good copy from R2, if any
        ok = False
        if os.path.exists(p):
            try:
                with Image.open(p) as im:
                    im.load()
                ok = True
            except Exception:
                os.remove(p)
        print("  ", rel, "- restored from R2" if ok else "- removed (will be rebuilt)")
    for t in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.tmp")):
        os.remove(t)
    if os.path.exists(cfg["done_csv"]):
        rows = list(csv.DictReader(open(cfg["done_csv"], newline="", encoding="utf-8")))
        keep = [r for r in rows if not r["status"].startswith("ERROR")]
        with open(cfg["done_csv"], "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS); w.writeheader(); w.writerows(keep)
        print(len(rows) - len(keep), "failed sheets put back on the to-do list:", [r["sheet"] for r in rows if r["status"].startswith("ERROR")])


# ---- SharePoint folder access (for packages uploaded as single files, not as one archive) ----
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"

class SharePointFolder:
    """Lists and downloads the files of a folder shared with an 'anyone with the link' sharing link."""
    def __init__(self, share_url):
        self.share_url = share_url
        self.login()

    def login(self):
        from urllib.parse import urlparse, parse_qs
        self.s = requests.Session(); self.s.headers["User-Agent"] = UA
        r = self.s.get(self.share_url, timeout=60)
        u = urlparse(r.url); q = parse_qs(u.query)
        if r.status_code != 200 or "id" not in q:
            raise RuntimeError(f"The sharing link did not open a folder (status {r.status_code}, landed on {r.url[:150]}). "
                               "Check that SHARE is the link of the folder itself.")
        self.root = q["id"][0]                                  # e.g. /personal/<user>/Documents/.../pc60_firstMS
        self.site = f"{u.scheme}://{u.netloc}" + "/".join(self.root.split("/")[:3])

    def _api(self, path, what):
        from urllib.parse import quote
        url = (f"{self.site}/_api/web/GetFolderByServerRelativePath(decodedurl='{quote(path.replace(chr(39), chr(39)*2))}')/{what}"
               "?$top=5000")
        for attempt in range(4):
            r = self.s.get(url, headers={"Accept": "application/json;odata=nometadata"}, timeout=60)
            if r.status_code == 200:
                return r.json()["value"]
            if r.status_code in (401, 403):
                self.login()
            time.sleep(3)
        raise RuntimeError(f"SharePoint did not list {path} (status {r.status_code}): {r.text[:300]}")

    def files(self, sub=""):
        """[(name, size, server_relative_url)] of the files directly in root/sub."""
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted((f["Name"], int(f["Length"]), f["ServerRelativeUrl"]) for f in self._api(p, "Files"))

    def folders(self, sub=""):
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted(f["Name"] for f in self._api(p, "Folders") if f["Name"] != "Forms")

    def download(self, rel_url, size, dest):
        from urllib.parse import quote
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            return
        url = f"{self.site}/_layouts/15/download.aspx?SourceUrl={quote(rel_url)}"
        err = None
        for attempt in range(5):
            try:
                with self.s.get(url, stream=True, timeout=120) as r:
                    if r.status_code in (401, 403):
                        self.login(); raise IOError(f"status {r.status_code}")
                    r.raise_for_status()
                    with open(dest + ".part", "wb") as f:
                        for chunk in r.iter_content(4 * 1024**2):
                            f.write(chunk)
                if os.path.getsize(dest + ".part") != size:
                    raise IOError("size mismatch")
                os.replace(dest + ".part", dest); return
            except Exception as e:
                err = e; time.sleep(5 * (attempt + 1))
        raise RuntimeError(f"could not download {rel_url}: {err}")

    def download_many(self, items, dest_dir, threads=8):
        """items: [(name, size, rel_url)] -> dest_dir/name."""
        from concurrent.futures import ThreadPoolExecutor
        os.makedirs(dest_dir, exist_ok=True)
        with ThreadPoolExecutor(threads) as ex:
            list(ex.map(lambda it: self.download(it[2], it[1], os.path.join(dest_dir, it[0])), items))

    def download_tree(self, sub, dest_dir, threads=16):
        """Download root/sub with all its subfolders (for the small .ini files). Returns the number of files."""
        n = 0
        items = self.files(sub)
        self.download_many(items, dest_dir, threads); n += len(items)
        for d in self.folders(sub):
            n += self.download_tree(f"{sub}/{d}", os.path.join(dest_dir, d), threads)
        return n

    def find_tifs(self, sub="", skip=("raw",), depth=4):
        """All .tif files under root/sub (subfolders included, folders named in skip left out):
        [(relative_path, size, server_relative_url)]."""
        out = [((sub + "/" if sub else "") + n, size, url) for n, size, url in self.files(sub) if n.lower().endswith((".tif", ".tiff"))]
        if depth > 0:
            for d in self.folders(sub):
                if d.lower() not in skip:
                    out += self.find_tifs((sub + "/" if sub else "") + d, skip, depth - 1)
        return out


In [ ]:
%%writefile /content/work/banat_frame.py
"""Map area of a First Military Survey sheet of the Banat of Temesvár. These sheets have one thick outer frame that
also encloses the title strip (top) and the column with place names (right); the map is the part below the title
line and left of the column line. Sheets differ in shape, so no fixed width/height ratio is assumed."""
import numpy as np, rasterio, warnings
warnings.filterwarnings('ignore')

def _best(prof, lo, hi):
    lo, hi = max(0, int(lo)), min(len(prof), int(hi))
    if hi <= lo: return None, 0.0
    i = lo + int(np.argmax(prof[lo:hi])); return i, float(prof[i])

def find_map_area(path, step=4):
    """-> dict(W, H, x=(x0, x1), y=(y0, y1), outer=(l, r, t, b), score) in scan pixels, or x/y None."""
    with rasterio.open(path) as ds:
        a = ds.read([1, 2, 3], out_shape=(3, ds.height // step, ds.width // step)).astype(np.int16)
        W, H = ds.width, ds.height
    dark = a.max(axis=0) < 110; h, w = dark.shape
    col = dark.mean(axis=0); row = dark.mean(axis=1)
    l, sl = _best(col, 0, 0.15 * w); r, sr = _best(col, 0.85 * w, w)
    t, st = _best(row, 0, 0.15 * h); b, sb = _best(row, 0.85 * h, h)
    out = dict(W=W, H=H, x=None, y=None, outer=None, score=None)
    if None in (l, r, t, b) or min(sl, sr, st, sb) < 0.45: return out
    # profiles between the outer lines only
    inner = dark[t + 3:b - 2, l + 3:r - 2]
    c2 = inner.mean(axis=0); r2 = inner.mean(axis=1)
    ti, sti = _best(r2, 150 // step, 1000 // step)                     # title line: first strong line below the top
    ci = None
    win = c2[(r - l) - 1500 // step:(r - l) - 250 // step]
    if len(win):
        thr = 0.6 * win.max()
        ci = (r - l) - 1500 // step + int(np.argmax(win >= thr)); sci = float(c2[ci])   # leftmost strong line of the name column
    if ti is None or ci is None or sti < 0.45 or sci < 0.3: return out
    x0, x1, y0, y1 = l, l + 3 + ci, t + 3 + ti, b
    out.update(x=(x0 * step, x1 * step), y=(y0 * step, y1 * step), outer=(l * step, r * step, t * step, b * step),
               score=round(min(sl, sr, st, sb, sti, sci), 2))
    return out

if __name__ == '__main__':
    import sys
    for p in sys.argv[1:]:
        print(p.split('/')[-1], find_map_area(p))


In [ ]:
%%writefile /content/work/temes_grid_cells.py
"""Sheet name -> (column, row) of the First Military Survey of the Banat of Temesvár (1769-1772, 208 sheets + 5 1/2),
read from the overview sheet Temes-attekinto (numbers written in the top-left corner of each cell, row by row)."""
ROWS = {  # row: (first sheet number, first column, number of sheets)
    0: (1, 2, 2), 1: (3, 1, 3), 2: (6, 1, 5), 3: (11, 1, 7), 4: (18, 1, 8), 5: (26, 1, 10), 6: (36, 0, 12), 7: (48, 0, 13),
    8: (61, 0, 13), 9: (74, 1, 12), 10: (86, 1, 12), 11: (98, 0, 13), 12: (111, 0, 13), 13: (124, 0, 13), 14: (137, 0, 13),
    15: (150, 0, 13), 16: (163, 1, 12), 17: (175, 1, 11), 18: (186, 5, 7), 19: (193, 6, 5), 20: (198, 6, 5), 21: (203, 7, 4),
    22: (207, 7, 2)}
CELL = {}
for r, (n0, c0, k) in ROWS.items():
    for i in range(k):
        CELL[f'{n0 + i:03d}'] = (c0 + i, r)
CELL['005-1_2'] = (4, 1)          # the extra half sheet "5 1/2" east of sheet 5
assert len(CELL) == 209 and CELL['208'] == (8, 22) and CELL['060'] == (12, 7) and CELL['185'] == (11, 17)


In [ ]:
%%writefile /content/work/temes_grid_fit.json
{
 "local": "+proj=aeqd +lat_0=45.5 +lon_0=21.3 +datum=WGS84 +units=m",
 "cell": [
  18206.244864,
  12137.496576
 ],
 "affine": [
  [
   0.8292831134438232,
   0.24817783165662424
  ],
  [
   -0.25840439063984266,
   0.8199385520268032
  ],
  [
   -126261.72566378553,
   75819.06021545887
  ]
 ],
 "similarity": [
  0.8254307765121512,
  0.25653400474445087,
  -125581.37224839,
  75580.66198408714
 ]
}

In [ ]:
%%writefile /content/work/temes_cp_tool.py
"""Control-point crops for the First Military Survey of the Banat of Temesvár: for a list of towns, find the sheet and the
approximate pixel (from the current grid fit), cut a crop with a labelled pixel grid and pack the crops into contact sheets."""
import json, math, os, numpy as np, rasterio, warnings
from pyproj import Transformer
from rasterio.windows import Window
from PIL import Image, ImageDraw
warnings.filterwarnings('ignore')
from temes_grid_cells import CELL
from banat_frame import find_map_area as find_frame
FIT = json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'temes_grid_fit.json')))
CW, CH = FIT['cell']; SOL = np.array(FIT['affine']); M = SOL[:2].T; T0 = SOL[2]; MINV = np.linalg.inv(M)
TL = Transformer.from_crs("EPSG:4326", FIT['local'], always_xy=True)
INV = {v: k for k, v in CELL.items()}
TOWNS = [("Temesvar cathedral",45.7577,21.2289),("Lugos",45.6886,21.9031),("Lippa",46.0886,21.6936),("Nagyszentmiklos",46.0722,20.6297),
 ("Csanad",46.1333,20.5833),("Zsombolya",45.7914,20.7172),("Nagykikinda",45.8297,20.4653),("Torokbecse",45.6000,20.0333),
 ("Nagybecskerek",45.3800,20.3900),("Pancsova",44.8708,20.6403),("Feherttemplom",44.8925,21.4178),("Versec",45.1167,21.3036),
 ("Ujpalanka",44.8467,21.3361),("Kevevara",44.7475,20.9761),("Antalfalva",45.1117,20.6214),("Alibunar",45.0806,20.9658),
 ("Perjamos",46.0500,20.8667),("Csene",45.7167,20.9000),("Detta",45.3906,21.2208),("Csak",45.5083,21.1286),("Buzias",45.6492,21.6036),
 ("Facsad",45.8478,22.1833),("Karansebes",45.4144,22.2208),("Mehadia",44.9050,22.3667),("Orsova",44.7254,22.3961),
 ("Bokszanbanya",45.3747,21.7097),("Resica",45.3008,21.8892),("Oravica",45.0400,21.6850),("Szaszkabanya",44.8867,21.7083),
 ("Omoldova",44.7231,21.6292),("Temesrekas",45.8500,21.5000),("Ecska",45.3200,20.4400),("Begaszentgyorgy",45.4850,20.5500),
 ("Vinga",46.0167,21.2000),("Varjas",45.9700,21.0400),("Torontalvasarhely",45.0700,20.6000),("Novi Knezevac",46.0444,20.1000),
 ("Ujszeged",46.2490,20.1600),("Belgrade fortress opposite (Pancsova bank)",44.8400,20.4600)]

def locate(lat, lon):
    """-> (sheet number or None, fx, fy) from the current grid fit."""
    X, Y = TL.transform(lon, lat); U, V = MINV @ (np.array([X, Y]) - T0)
    u, v = U / CW, -V / CH; c, r = math.floor(u), math.floor(v)
    return INV.get((c, r)), u - c, v - r

def crop(path, cx, cy, label, half=1700, size=640):
    with rasterio.open(path) as ds:
        x0 = int(max(0, min(ds.width - 2*half, cx - half))); y0 = int(max(0, min(ds.height - 2*half, cy - half)))
        a = ds.read([1, 2, 3], window=Window(x0, y0, 2*half, 2*half), out_shape=(3, size, size))
    im = Image.fromarray(a.transpose(1, 2, 0)); d = ImageDraw.Draw(im); k = size / (2*half)
    for v in range((x0 // 250 + 1) * 250, x0 + 2*half, 250):
        d.line([((v-x0)*k, 0), ((v-x0)*k, size)], fill=(255, 0, 0) if v % 1000 == 0 else (255, 170, 170), width=1)
        if v % 500 == 0: d.text(((v-x0)*k + 2, 2), str(v), fill=(255, 0, 0))
    for v in range((y0 // 250 + 1) * 250, y0 + 2*half, 250):
        d.line([(0, (v-y0)*k), (size, (v-y0)*k)], fill=(0, 0, 255) if v % 1000 == 0 else (170, 170, 255), width=1)
        if v % 500 == 0: d.text((2, (v-y0)*k + 2), str(v), fill=(0, 0, 255))
    d.rectangle([0, size-15, 8*len(label)+8, size], fill=(255, 255, 255)); d.text((4, size-13), label, fill=(0, 0, 0))
    return im

def plan():
    """[(town, lat, lon, sheet, fx, fy)] for towns that fall on a sheet."""
    out = []
    for name, lat, lon in TOWNS:
        s, fx, fy = locate(lat, lon)
        if s: out.append((name, lat, lon, s, fx, fy))
    return out

def make(sheet_path, out_dir, per_sheet=6, cols=3):
    """sheet_path(n) -> local path of sheet n (or None). Writes contact sheets + points.json into out_dir."""
    os.makedirs(out_dir, exist_ok=True); tiles = []; info = []
    for name, lat, lon, s, fx, fy in plan():
        p = sheet_path(s)
        if not p: info.append(dict(town=name, sheet=s, status='sheet not available')); continue
        fr = find_frame(p)
        if not (fr['x'] and fr['y']): info.append(dict(town=name, sheet=s, status='frame not found', size=[fr['W'], fr['H']])); continue
        px = fr['x'][0] + fx*(fr['x'][1]-fr['x'][0]); py = fr['y'][0] + fy*(fr['y'][1]-fr['y'][0])
        tiles.append(crop(p, px, py, f"{len(tiles)+1}. {name} / sheet {s}"))
        info.append(dict(n=len(tiles), town=name, lat=lat, lon=lon, sheet=s, frame_x=fr['x'], frame_y=fr['y'], pred_px=[int(px), int(py)], status='ok'))
    size = tiles[0].size[0] if tiles else 640
    for g in range(0, len(tiles), per_sheet):
        grp = tiles[g:g+per_sheet]; rows = math.ceil(len(grp)/cols)
        sheet = Image.new('RGB', (cols*size, rows*size), 'white')
        for i, t in enumerate(grp): sheet.paste(t, ((i % cols)*size, (i//cols)*size))
        sheet.save(os.path.join(out_dir, f'cp_{g//per_sheet+1:02d}.jpg'), quality=88)
    json.dump(info, open(os.path.join(out_dir, 'points.json'), 'w'), indent=1)
    return info


## Run
`SHARE` is the sharing link of the `pc35-firstMS_Trans_Temes` folder.

In [ ]:
SHARE = "PASTE_THE_SHARING_LINK_OF_THE_FOLDER_HERE"
RAW = "1ms_Geotiff_export_Temesvar/raw"

import danfront_funcs as F, temes_cp_tool as E, collections, shutil
SP = F.SharePointFolder(SHARE)
print("shared folder:", SP.root.split("/")[-1], "| contains:", SP.folders())
files = {n: (n, size, url) for n, size, url in SP.files(RAW)}
plan = E.plan()
need = sorted({f"{p[3]}.tif" for p in plan})
have = [n for n in need if n in files]
print(len(plan), "towns on", len(need), "sheets;", len(have), "of them found,", round(sum(files[n][1] for n in have)/1e9, 1), "GB")
SP.download_many([files[n] for n in have], "/content/sheets", threads=6)
info = E.make(lambda n: f"/content/sheets/{n}.tif" if os.path.exists(f"/content/sheets/{n}.tif") else None, "/content/cp_out")
print(collections.Counter(i["status"] for i in info))
for i in info:
    if i["status"] != "ok": print("  ", i["town"], "sheet", i["sheet"], "-", i["status"])
shutil.make_archive("/content/temes_control_points", "zip", "/content/cp_out")
from google.colab import files as gfiles
gfiles.download("/content/temes_control_points.zip")